# Clustered data: overconfidence and partial pooling

Companion to the course pages *Clustered and hierarchical data* and *How to: analyze clustered data*.

Mia studies a gene variant that, to keep the simulation simple, is carried by whole families (in real families genotype also varies between relatives, but relatives still resemble each other) and measures LDL cholesterol in 20 relatives per family. The outcome is in arbitrary units. Relatives in the same family resemble each other. We simulate this many times from a known truth and ask:

1. How often is a 95% interval right, and how often does it wrongly exclude zero when there is **no** effect, for each way of analyzing the data?
2. How does the problem grow with how different the families are?
3. What does **partial pooling** do to the estimates for small and large clusters?

Change the parameters in the next cell and use *Run All*. Needs `numpy`, `pandas`, `matplotlib`, `statsmodels`.

In [ ]:
# ---- Parameters: change these and re-run ----
N_CLUSTERS = 15        # number of families (half are carrier families)
PER_CLUSTER = 20       # relatives per family
CLUSTER_SD = 3.0       # how much families differ from each other (SD of the family effect)
NOISE_SD = 3.0         # how much relatives differ within a family
TRUE_EFFECT = 0.0      # true effect of carrying the variant; 0 lets us count false positives
N_SIMS = 300           # simulated datasets per setting
SEED = 0

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")   # mixed models can warn about boundary fits in some simulated datasets
rng = np.random.default_rng(SEED)


def simulate_clusters(n_clusters=N_CLUSTERS, per_cluster=PER_CLUSTER, cluster_sd=CLUSTER_SD):
    family = np.repeat(np.arange(n_clusters), per_cluster)
    carrier_family = rng.permutation(np.arange(n_clusters) % 2)     # carrier status is set per family
    x = carrier_family[family]
    y = TRUE_EFFECT * x + rng.normal(0, cluster_sd, n_clusters)[family] + rng.normal(0, NOISE_SD, len(family))
    return pd.DataFrame({"family": family, "x": x, "y": y})


def intervals(df, include_mixed=True):
    """95% CI for the carrier effect under each analysis."""
    models = {
        "ignoring clusters": smf.ols("y ~ x", df).fit(),
        "cluster-robust SE": smf.ols("y ~ x", df).fit(cov_type="cluster", cov_kwds={"groups": df["family"]}, use_t=True),
        "analysis of cluster means": smf.ols("y ~ x", df.groupby(["family", "x"], as_index=False)["y"].mean()).fit(),
    }
    if include_mixed:
        models["multilevel model"] = smf.mixedlm("y ~ x", df, groups=df["family"]).fit()
    return {name: tuple(model.conf_int().loc["x"]) for name, model in models.items()}

## 1. How trustworthy is each analysis?

For each simulated study we compute the 95% interval for the carrier effect four ways. A trustworthy method has **coverage close to 95%** and, when the true effect is 0, **excludes 0 in about 5% of studies**.

In [ ]:
records = []
for _ in range(N_SIMS):
    for name, (low, high) in intervals(simulate_clusters()).items():
        records.append({"method": name, "covers truth": low <= TRUE_EFFECT <= high,
                        "excludes 0": not (low <= 0 <= high), "width": high - low})
summary = pd.DataFrame(records).groupby("method").mean().round(2)
print(f"True effect = {TRUE_EFFECT}; {N_CLUSTERS} families x {PER_CLUSTER} relatives; {N_SIMS} simulated studies")
summary.loc[["ignoring clusters", "cluster-robust SE", "multilevel model", "analysis of cluster means"]]

## 2. The problem grows with how different the families are

Same study, but varying `CLUSTER_SD`. When families are identical (SD 0) clustering is irrelevant; the more families differ, the more a naive analysis overstates its precision.

In [ ]:
rows = []
for sd in [0, 1, 2, 3, 5]:
    excluded = {"ignoring clusters": [], "cluster-robust SE": [], "analysis of cluster means": []}
    for _ in range(N_SIMS):
        for name, (low, high) in intervals(simulate_clusters(cluster_sd=sd), include_mixed=False).items():
            excluded[name].append(not (low <= 0 <= high))
    icc = sd**2 / (sd**2 + NOISE_SD**2)
    rows.append({"CLUSTER_SD": sd, "ICC": round(icc, 2), **{k: round(np.mean(v), 2) for k, v in excluded.items()}})
print("Share of studies whose interval excludes 0 (should be about 0.05 when the true effect is 0):")
pd.DataFrame(rows).set_index("CLUSTER_SD")

## 3. Partial pooling: small clusters get pulled toward the average

Now 25 families of very uneven size (2 to 40 relatives). The *raw* family mean is the no-pooling estimate; the multilevel model's estimate shrinks each family's mean toward the overall mean, more for small, noisy families. We know each family's true mean here, so we can check which estimate is closer.

In [ ]:
n_families = 25


def draw_families():
    """One simulated set of families of very uneven size, with raw and partially pooled family means."""
    sizes = rng.integers(2, 41, n_families)
    true_means = 20 + rng.normal(0, CLUSTER_SD, n_families)
    family = np.repeat(np.arange(n_families), sizes)
    data = pd.DataFrame({"family": family, "y": true_means[family] + rng.normal(0, NOISE_SD, len(family))})
    raw = data.groupby("family")["y"].mean().to_numpy()
    fit = smf.mixedlm("y ~ 1", data, groups=data["family"]).fit()
    shrunk = np.array([fit.fe_params["Intercept"] + fit.random_effects[g].iloc[0] for g in range(n_families)])
    return sizes, true_means, raw, shrunk, data["y"].mean()


# Average the comparison over many simulated sets of families: a single draw can go either way.
errors = []
for _ in range(N_SIMS // 3):
    _, truth, raw_means, shrunk_means, _ = draw_families()
    errors.append((np.mean((raw_means - truth) ** 2), np.mean((shrunk_means - truth) ** 2)))
errors = np.array(errors)
print(f"RMSE vs. the true family means, averaged over {len(errors)} simulations:")
print(f"  no pooling (raw family means): {np.sqrt(errors[:, 0].mean()):.2f}")
print(f"  partial pooling:               {np.sqrt(errors[:, 1].mean()):.2f}")

sizes, true_means, raw, shrunk, overall = draw_families()    # one more draw, for the plot
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(sizes, raw - overall, label="raw family mean (no pooling)")
ax.scatter(sizes, shrunk - overall, marker="x", label="partial pooling")
ax.axhline(0, color="gray", linestyle=":")
ax.set_xlabel("relatives measured in the family")
ax.set_ylabel("estimate minus the overall mean")
ax.legend()
plt.show()

## Things to try

- Set `CLUSTER_SD = 0`: all four methods behave, because there is no clustering to ignore.
- Set `N_CLUSTERS = 6`: the honest methods stay valid, but their intervals become very wide (about 10 units, versus about 6 with 15 families), because the effective sample is six families.
- Set `N_CLUSTERS = 60`: the honest intervals shrink, but ignoring clusters still excludes 0 in roughly half of the studies. The problem doesn't go away with more families; it is a property of the analysis.
- Set `TRUE_EFFECT = 2`: the `excludes 0` column is now the power to detect the effect. The naive analysis looks far more powerful (about 70%) than the honest ones (about 20%), and the difference is false confidence.
- In section 3, the points for small families move furthest toward zero, and the partial-pooling RMSE should be lower than the raw-means RMSE. Increase `NOISE_SD` and the pooling effect gets stronger.